<a href="https://colab.research.google.com/github/NismaK/ML_Homeworkbook/blob/main/ML_zoomcamp_homework_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd, numpy as np

In [ ]:
cols = ['engine_displacement','horsepower','vehicle_weight','model_year','fuel_efficiency_mpg']
df = pd.read_csv('car_fuel_efficiency_2026.csv')[cols]

Question 1

There's one column with missing values. What is it?

'engine_displacement'

'horsepower'

'vehicle_weight'

'model_year'

In [ ]:
print(df.isnull().sum())                       # Q1

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64


Question 2

What's the median (50% percentile) for variable 'horsepower'?

204

254

304

354

In [ ]:
print(df.horsepower.median())                  # Q2


254.0


In [ ]:
def split(df, seed):
    n = len(df); n_val = int(n*0.2); n_test = int(n*0.2); n_train = n - n_val - n_test
    np.random.seed(seed)
    idx = np.arange(n); np.random.shuffle(idx)
    return df.iloc[idx[:n_train]], df.iloc[idx[n_train:n_train+n_val]], df.iloc[idx[n_train+n_val:]]


In [ ]:

def train(X, y, r=0):
    X = np.column_stack([np.ones(len(X)), X])
    XTX = X.T @ X + r * np.eye(X.shape[1])
    w = np.linalg.inv(XTX) @ X.T @ y
    return w[0], w[1:]

def rmse(y, p):
    return np.sqrt(((y - p)**2).mean())

def prep(d, fill):
    return d.drop(columns='fuel_efficiency_mpg').fillna(fill).values, d.fuel_efficiency_mpg.values

def run(seed, fill_mode, r=0):
    tr, va, te = split(df, seed)
    fill = 0 if fill_mode == '0' else tr.horsepower.mean()   # mean from train only
    Xt, yt = prep(tr, fill); Xv, yv = prep(va, fill)
    w0, w = train(Xt, yt, r)
    return rmse(yv, w0 + Xv @ w)

Question 3

Which option gives better RMSE?

Options:


With 0

With mean

Both are equally good

In [ ]:
# Q3
print(round(run(42,'0'),3), round(run(42,'mean'),3))

2.205 2.202


Question 4

Which r gives the best RMSE?

If multiple options give the same best RMSE, select the smallest r.

Options:

0

0.01

0.1

1

5

10

100

In [ ]:
# Q4
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    print(r, round(run(42,'0',r),4))


0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292


Question 5

What's the value of std?

0.006

0.016

0.029

0.036

In [ ]:
# Q5
print(round(np.std([run(s,'0') for s in range(10)]), 3))

0.029


Question 6

What's the RMSE on the test dataset?

Options:

0.236

2.236

22.10

221.0

In [ ]:
# Q6
tr, va, te = split(df, 9)
full = pd.concat([tr, va])
Xf, yf = prep(full, 0); Xte, yte = prep(te, 0)
w0, w = train(Xf, yf, 0.001)
print(round(rmse(yte, w0 + Xte @ w), 3))

2.236
